# BERT-QPP$_{cross}$ on TREC DL 2019 / 2020

Runs a trained BERT-QPP$_{cross}$ checkpoint over pre-existing run files (already on Drive, no live retrieval) for 8 rankers, and correlates predicted QPP scores against actual MAP@50, nDCG@100, and nDCG@10 (via `pytrec_eval`) for:
- TREC DL 2019 alone
- TREC DL 2020 alone
- TREC DL 2019 + 2020 pooled

No Pyserini/PyTerrier and no live indexing — run files, queries, and qrels are read directly from Drive. The only external fetch is the MS MARCO passage collection (needed to get each top-ranked document's text for the cross-encoder input), cached on Drive after the first run.

## 1. Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers pytrec_eval scipy pandas tqdm

## 2. Configuration — edit paths to match your Drive layout

In [ ]:
import os

drive_base_runfiles_19 = "/content/drive/MyDrive/precise-qpp/data/TRECDL19-runfiles"
drive_base_runfiles_20 = "/content/drive/MyDrive/precise-qpp/data/TRECDL20-runfiles"
drive_base_queries_qrels = "/content/drive/MyDrive/precise-qpp/data/TREC-queries-qrels"

MODEL_PATH = "/content/drive/MyDrive/precise-qpp/models/tuned_model-ce_bert-base-uncased_e1_b8"  # edit me: pre-existing trained checkpoint

# MS MARCO passage collection, needed for doc text lookup. Downloaded once and cached here if missing.
COLLECTION_PATH = "/content/drive/MyDrive/precise-qpp/data/collection.tsv"  # edit me if you already have it cached elsewhere

OUT_DIR = "/content/drive/MyDrive/precise-qpp/results/bertqpp_cross_dl1920"
os.makedirs(OUT_DIR, exist_ok=True)

SYSTEM_NAMES = ["BM25", "rm3", "colbert.e2e", "e5", "monot5", "prf_rank_beta05", "splade", "prf_rerank_beta05"]

def run_files_for_year(base_dir, year):
    """Build the fixed list of per-ranker run files for TREC DL `year` ('2019' or '2020')."""
    yy = year[-2:]
    return [
        f"{base_dir}/BM25.{year}.100.res",
        f"{base_dir}/rm3.100.res",
        f"{base_dir}/colbert.e2e.100.res",
        f"{base_dir}/e5_dl_{yy}.100.res",
        f"{base_dir}/monot5.100.res",
        f"{base_dir}/prf_rank_beta05.{year}.100.res",
        f"{base_dir}/splade.100.res",
        f"{base_dir}/prf_rerank_beta05.{year}.100.res",
    ]

def qrels_path(year):
    return f"{drive_base_queries_qrels}/pass_{year}.qrels"

def queries_path(year):
    return f"{drive_base_queries_qrels}/pass_{year}.queries"

YEAR_RUN_DIRS = {"2019": drive_base_runfiles_19, "2020": drive_base_runfiles_20}

## 3. Parsers for queries / qrels / TREC run files

In [ ]:
import pytrec_eval
from collections import defaultdict

def parse_queries(path):
    queries = {}
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            qid, text = line.split('\t', 1)
            queries[qid] = text
    return queries

def parse_qrels(path):
    with open(path) as f:
        return pytrec_eval.parse_qrel(f)

def parse_run(path):
    """TREC 6-col run file -> (run_scores: {qid: {docid: score}}, top1_doc: {qid: docid}).
    top1 is picked by lowest rank column, not by file order, so it's robust to unsorted runs."""
    run_scores = defaultdict(dict)
    best_rank = {}
    top1 = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            qid, _, docid, rank, score, _ = parts[:6]
            run_scores[qid][docid] = float(score)
            rank = int(rank)
            if qid not in best_rank or rank < best_rank[qid]:
                best_rank[qid] = rank
                top1[qid] = docid
    return dict(run_scores), top1

## 4. Load queries, qrels, and every system's run file for both years

In [ ]:
year_queries = {}
year_qrels = {}
run_scores_by_system = {}   # (year, system) -> {qid: {docid: score}}
top1_by_system = {}         # (year, system) -> {qid: docid}
needed_docids = set()

for year, base_dir in YEAR_RUN_DIRS.items():
    year_queries[year] = parse_queries(queries_path(year))
    year_qrels[year] = parse_qrels(qrels_path(year))
    print(f"DL{year[-2:]}: {len(year_queries[year])} queries, {len(year_qrels[year])} judged")

    for system, run_file in zip(SYSTEM_NAMES, run_files_for_year(base_dir, year)):
        if not os.path.exists(run_file) or os.path.getsize(run_file) == 0:
            print(f"  [SKIP] {system}: missing or empty ({run_file})")
            continue
        run_scores, top1 = parse_run(run_file)
        run_scores_by_system[(year, system)] = run_scores
        top1_by_system[(year, system)] = top1
        needed_docids.update(top1.values())
        print(f"  [OK]   {system}: {len(run_scores)} queries")

print(f"\nNeed text for {len(needed_docids)} unique top-1 docids across both years/systems")

## 5. Resolve top-1 document text
Downloads and caches the MS MARCO collection on Drive once, then does a single sequential scan pulling out only the docids actually needed (a few hundred, not all 8.8M passages) — keeps memory use small.

In [ ]:
if not os.path.exists(COLLECTION_PATH):
    os.makedirs(os.path.dirname(COLLECTION_PATH), exist_ok=True)
    !wget -q --show-progress -O /content/collectionandqueries.tar.gz \
        https://msmarco.blob.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
    !tar -xzf /content/collectionandqueries.tar.gz -C /content collection.tsv
    !mv /content/collection.tsv "$COLLECTION_PATH"
    !rm /content/collectionandqueries.tar.gz
else:
    print(f"[INFO] Using cached collection at {COLLECTION_PATH}")

doc_text = {}
with open(COLLECTION_PATH) as f:
    for line in f:
        docid, text = line.rstrip('\n').split('\t', 1)
        if docid in needed_docids:
            doc_text[docid] = text
            if len(doc_text) == len(needed_docids):
                break

missing = needed_docids - doc_text.keys()
print(f"Resolved {len(doc_text)}/{len(needed_docids)} docids ({len(missing)} missing)")

## 6. Predict QPP scores with the BERT-QPP$_{cross}$ checkpoint

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm import tqdm

model = CrossEncoder(MODEL_PATH, num_labels=1)

predictions = {}  # (year, system) -> {qid: predicted_score}
for (year, system), top1 in tqdm(top1_by_system.items()):
    qids = [qid for qid in top1 if top1[qid] in doc_text]
    pairs = [[year_queries[year][qid], doc_text[top1[qid]]] for qid in qids]
    scores = model.predict(pairs, show_progress_bar=False)
    predictions[(year, system)] = {qid: float(s) for qid, s in zip(qids, scores)}

## 7. Actual per-query effectiveness (MAP@50, nDCG@100, nDCG@10) via `pytrec_eval`

In [ ]:
METRICS = {"map_cut.50", "ndcg_cut.100", "ndcg_cut.10"}
METRIC_KEYS = {"MAP@50": "map_cut_50", "nDCG@100": "ndcg_cut_100", "nDCG@10": "ndcg_cut_10"}

actual_scores_by_system = {}  # (year, system) -> {qid: {metric_key: value}}
for (year, system), run_scores in run_scores_by_system.items():
    evaluator = pytrec_eval.RelevanceEvaluator(year_qrels[year], METRICS)
    actual_scores_by_system[(year, system)] = evaluator.evaluate(run_scores)

## 8. Correlate predicted vs. actual — DL19, DL20, and DL19+20 pooled

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr, kendalltau

SCOPES = {"DL19": ["2019"], "DL20": ["2020"], "DL19+20": ["2019", "2020"]}

rows = []
for system in SYSTEM_NAMES:
    for scope, years in SCOPES.items():
        pred, actual = {}, {}
        for year in years:
            key = (year, system)
            if key not in predictions:
                continue
            pred.update(predictions[key])
            actual.update(actual_scores_by_system[key])
        if not pred:
            continue
        common = [qid for qid in pred if qid in actual]
        p = [pred[qid] for qid in common]
        for metric_label, metric_key in METRIC_KEYS.items():
            a = [actual[qid][metric_key] for qid in common]
            rows.append({
                "system": system,
                "scope": scope,
                "metric": metric_label,
                "n_queries": len(common),
                "pearson": pearsonr(p, a)[0],
                "spearman": spearmanr(p, a)[0],
                "kendall": kendalltau(p, a)[0],
            })

results_df = pd.DataFrame(rows)
results_df

## 9. Save results

In [ ]:
csv_path = f"{OUT_DIR}/correlations.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved correlation table to {csv_path}")

pivot = results_df.pivot_table(index=["system", "scope"], columns="metric", values="pearson")
pivot